# 02 · TF-IDF y n-gramas

**Duración estimada:** 25 minutos

## Propósito

En la tabla del `01`, todas las palabras pesan igual: `carne`, que aparece en un
solo documento, cuenta lo mismo que `el`, que aparece en todos.

TF-IDF es una manera de darle a cada palabra un peso según cuán informativa es.
Vamos a calcularlo a mano, después con la librería, **y a comparar los dos
resultados**.

## Qué consume

Nada. Las mismas cuatro oraciones.

## 1. La idea

TF-IDF multiplica dos cosas.

**TF** — *term frequency*. Cuánto aparece la palabra **en este documento**. Si
aparece mucho acá, probablemente el documento hable de eso.

**IDF** — *inverse document frequency*. Cuán rara es la palabra **en todo el
conjunto**. Si aparece en todos los documentos, no sirve para distinguirlos.

El peso final es `TF × IDF`. Una palabra pesa mucho cuando aparece **mucho acá y
poco en el resto**.

Las fórmulas más difundidas son estas:

```
TF  = veces que aparece la palabra en el documento
      ────────────────────────────────────────────
      total de palabras del documento

IDF = log( total de documentos
           ─────────────────────────────────────── )
           documentos que contienen la palabra
```

Vamos a implementarlas tal cual.

## 2. TF-IDF a mano

In [ ]:
# math: para el logaritmo.
import math

# pandas, para las tablas.
import pandas as pd

documentos = [
    "El perro muerde al hombre",
    "El hombre muerde al perro",
    "El perro come carne",
    "El hombre come comida",
]

# Tokenizamos igual que en el cuaderno 00.
documentos_en_tokens = []
for documento in documentos:
    documentos_en_tokens.append(documento.lower().split())

vocabulario = []
for tokens in documentos_en_tokens:
    for token in tokens:
        if token not in vocabulario:
            vocabulario.append(token)
vocabulario.sort()

print(vocabulario)

### El IDF de cada palabra

Se calcula una sola vez para todo el conjunto: no depende de qué documento estemos
mirando.

In [ ]:
total_documentos = len(documentos)

idf_de_cada_palabra = {}

for palabra in vocabulario:
    # ¿En cuántos documentos aparece esta palabra?
    documentos_que_la_contienen = 0
    for tokens in documentos_en_tokens:
        if palabra in tokens:
            documentos_que_la_contienen = documentos_que_la_contienen + 1
    idf = math.log(total_documentos / documentos_que_la_contienen)
    idf_de_cada_palabra[palabra] = idf

filas = []
for palabra in vocabulario:
    documentos_que_la_contienen = 0
    for tokens in documentos_en_tokens:
        if palabra in tokens:
            documentos_que_la_contienen = documentos_que_la_contienen + 1
    fila = {
        "palabra": palabra,
        "en_cuantos_docs": documentos_que_la_contienen,
        "idf": round(idf_de_cada_palabra[palabra], 3),
    }
    filas.append(fila)

pd.DataFrame(filas)

Leelo con atención, porque acá está lo que TF-IDF quiere hacer:

- `carne` aparece en **un** documento y tiene el IDF más alto: 1,386.
- `el` aparece en **los cuatro** y tiene IDF **cero**.

Un IDF de cero significa que esa palabra va a tener peso cero en todos los
documentos: TF × 0 = 0. **Una palabra que está en todos los documentos no sirve
para distinguir ninguno**, y la fórmula la anula sola.

Es elegante: TF-IDF descarta las stopwords sin necesidad de una lista escrita a
mano.

### El TF y el peso final

In [ ]:
def calcular_tfidf_del_documento(tokens):
    # Devuelve la lista de pesos TF-IDF, una posición por palabra del vocabulario.
    total_de_palabras = len(tokens)
    pesos = []
    for palabra in vocabulario:
        veces = tokens.count(palabra)
        tf = veces / total_de_palabras
        peso = tf * idf_de_cada_palabra[palabra]
        pesos.append(round(peso, 3))
    return pesos


pesos_a_mano = []
for tokens in documentos_en_tokens:
    pesos_a_mano.append(calcular_tfidf_del_documento(tokens))

nombres_de_fila = ["Doc 1", "Doc 2", "Doc 3", "Doc 4"]
tabla_a_mano = pd.DataFrame(pesos_a_mano, columns=vocabulario, index=nombres_de_fila)
tabla_a_mano

La columna `el` es toda ceros, como esperábamos. Y en el documento 3 —*El
perro come carne*— la palabra con más peso es `carne`.

Funciona.

## 3. Lo mismo con la librería

In [ ]:
# TfidfVectorizer funciona igual que CountVectorizer: fit para aprender,
# transform para aplicar.
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer()
matriz = vectorizador.fit_transform(documentos)

tabla_sklearn = pd.DataFrame(
    matriz.toarray().round(3),
    columns=vectorizador.get_feature_names_out(),
    index=nombres_de_fila,
)
tabla_sklearn

## 4. Comparemos

Pongamos las dos versiones del documento 3 una al lado de la otra.

In [ ]:
filas = []
for palabra in vocabulario:
    fila = {
        "palabra": palabra,
        "a_mano": tabla_a_mano.loc["Doc 3", palabra],
        "sklearn": tabla_sklearn.loc["Doc 3", palabra],
    }
    filas.append(fila)

pd.DataFrame(filas)

**No coincide ni un solo número.**

Y hay algo peor que el desacuerdo: en nuestra versión `el` vale **cero**, como
dice la fórmula. En la de sklearn vale **0,344**.

Si alguien hace la cuenta a mano y la compara con la salida de la librería, va a
concluir que se equivocó. No se equivocó.

**`TfidfVectorizer` no implementa la fórmula que acabamos de escribir.** Hace dos
cosas distintas, y las dos están documentadas.

### Diferencia 1: el IDF está suavizado

Miremos los IDF que calculó la librería.

In [ ]:
filas = []
posicion = 0
for palabra in vectorizador.get_feature_names_out():
    fila = {
        "palabra": palabra,
        "idf_a_mano": round(idf_de_cada_palabra[palabra], 3),
        "idf_sklearn": round(float(vectorizador.idf_[posicion]), 3),
    }
    filas.append(fila)
    posicion = posicion + 1

pd.DataFrame(filas)

Ninguno es cero. El mínimo es 1.

La fórmula de sklearn no es `log(N / df)` sino:

```
IDF = log( (1 + N) / (1 + df) ) + 1
```

Ese `+1` de arriba y abajo evita dividir por cero cuando una palabra no aparece en
ningún documento. Y el `+1` final hace que **ninguna palabra quede en cero**: la
librería decidió que una palabra presente en todos los documentos igual aporta
algo, en vez de anularla.

Es una decisión defendible. Pero es **otra decisión** que la de la fórmula del
pizarrón.

### Diferencia 2: cada fila está normalizada

Sumemos los cuadrados de cada fila de la tabla de sklearn y saquemos la raíz.

In [ ]:
# numpy: para la raíz cuadrada.
import numpy as np

# Usamos la matriz sin redondear, para que la cuenta salga exacta.
valores_sin_redondear = matriz.toarray()

posicion = 0
for nombre in nombres_de_fila:
    fila = valores_sin_redondear[posicion]
    largo = np.sqrt((fila ** 2).sum())
    print(nombre, "-> largo del vector:", round(float(largo), 3))
    posicion = posicion + 1

**Todas dan 1.**

sklearn divide cada fila por su propio largo, para que todos los documentos tengan
vectores del mismo tamaño. Se llama **normalización L2**, y sirve para que un
documento largo no pese más que uno corto solo por ser largo.

Es otra decisión razonable, y también invisible desde la fórmula.

### ¿Se puede pedirle la fórmula del pizarrón?

Los dos comportamientos tienen parámetros. `smooth_idf=False` quita el suavizado
del cociente y `norm=None` quita la normalización. Probemos.

In [ ]:
vectorizador_simple = TfidfVectorizer(smooth_idf=False, norm=None)
matriz_simple = vectorizador_simple.fit_transform(documentos)

tabla_simple = pd.DataFrame(
    matriz_simple.toarray().round(3),
    columns=vectorizador_simple.get_feature_names_out(),
    index=nombres_de_fila,
)
tabla_simple

Mirá la columna `el`. **Sigue sin ser cero: ahora vale exactamente 1.**

La razón es que la fórmula de sklearn, aun sin suavizado, es:

```
IDF = log( N / df ) + 1
```

Ese `+ 1` del final **no tiene parámetro que lo apague**. Está siempre. Para `el`,
`log(4/4)` da 0, y el `+1` lo deja en 1.

Comprobalo.

In [ ]:
for suavizado in [True, False]:
    prueba = TfidfVectorizer(smooth_idf=suavizado, norm=None)
    prueba.fit(documentos)
    posicion = list(prueba.get_feature_names_out()).index("el")
    print("smooth_idf =", suavizado, "-> idf de 'el' =", round(float(prueba.idf_[posicion]), 3))

print()
print("la fórmula del pizarrón daría:", round(math.log(4 / 4), 3))

**No hay manera de que `TfidfVectorizer` calcule la fórmula que vimos al
principio.** Se le parece, pero no es. Si necesitás exactamente esa fórmula —por
ejemplo, porque estás reproduciendo un trabajo publicado— la tenés que calcular
vos, como hicimos en la sección 2.

Y queda todavía otra diferencia: sklearn **no divide el TF por el total de
palabras del documento**, usa la cuenta cruda.

> **La lección de este cuaderno no es TF-IDF.**
>
> Es que **una librería implementa una versión de la fórmula, no la fórmula**, y
> que esa versión incorpora decisiones que cambian los resultados y que a veces
> ni siquiera se pueden desactivar.
>
> Cuando reportes un número salido de una librería, tenés que poder decir con qué
> herramienta y con qué parámetros lo calculaste. Si no, el número no es
> reproducible por nadie, ni siquiera por vos dentro de seis meses.

## 5. N-gramas

Queda pendiente el problema del `01`: los documentos 1 y 2 tienen el mismo vector
aunque digan cosas opuestas.

Un **n-grama** es una secuencia de n palabras seguidas. Contar bigramas —pares—
en vez de palabras sueltas recupera algo del orden.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# ngram_range=(1, 2) cuenta palabras sueltas Y pares de palabras.
contador_bigramas = CountVectorizer(ngram_range=(1, 2))
matriz_bigramas = contador_bigramas.fit_transform(documentos)

terminos = contador_bigramas.get_feature_names_out()

print("términos con palabras sueltas:", 8)
print("términos con palabras y pares:", len(terminos))
print()

solo_pares = []
for termino in terminos:
    if " " in termino:
        solo_pares.append(termino)

print("los pares:")
print(solo_pares)

In [ ]:
tabla_bigramas = pd.DataFrame(
    matriz_bigramas.toarray(),
    columns=terminos,
    index=nombres_de_fila,
)

fila_1 = tabla_bigramas.loc["Doc 1"]
fila_2 = tabla_bigramas.loc["Doc 2"]

print("Doc 1:", documentos[0])
print("Doc 2:", documentos[1])
print()
print("¿los vectores siguen siendo iguales?", list(fila_1) == list(fila_2))

**Ahora se distinguen.** `perro muerde` está en el documento 1 y no en el 2;
`hombre muerde` al revés.

El costo está a la vista: el vocabulario pasó de 8 términos a 19, con cuatro
oraciones. En un corpus real, agregar bigramas multiplica el vocabulario por diez
o más, y la mayoría de esos pares aparece una sola vez.

Y fijate qué recuperamos exactamente: sabemos que *perro* viene antes de *muerde*.
**No sabemos que el perro es el sujeto.** Es una pista de posición, no una
relación gramatical.

Eso último es lo que hace spaCy, y es la unidad que viene.

> ### Perilla
>
> Cambiá `ngram_range=(1, 2)` por `(1, 3)` y mirá cuánto crece el vocabulario.
> Después probá `(2, 2)`: solo pares, sin palabras sueltas.
>
> La pregunta no es cuál es mejor. Es: **¿qué pregunta estás haciendo?** Si te
> interesan locuciones fijas —*a la gorra*, *puesta en marcha*—, los pares y tríos
> sirven. Si te interesa el vocabulario temático, sobran.

In [ ]:
for rango in [(1, 1), (1, 2), (1, 3), (2, 2)]:
    contador = CountVectorizer(ngram_range=rango)
    contador.fit(documentos)
    cantidad = len(contador.get_feature_names_out())
    print("ngram_range =", rango, "->", cantidad, "términos")

## Lo que hiciste hoy

- Calculaste TF-IDF a mano, con la fórmula, y entendiste por qué anula las
  palabras que están en todos los documentos.
- Lo calculaste con la librería y **los números no coincidieron**.
- Encontraste las dos razones: el IDF suavizado y la normalización L2.
- Aprendiste que un número salido de una librería no es reproducible si no decís
  con qué parámetros lo calculaste.
- Viste que los n-gramas recuperan posición, pero no relaciones gramaticales.

## Lo que viene

El `03`, que es el último del teórico: ya sabemos armar la representación, así
que vamos a medir **dónde deja de servir**. Dos oraciones que dicen lo mismo con
otras palabras, y una palabra que significa tres cosas distintas.

## Glosario

| término | qué es |
|---|---|
| **token** | La unidad mínima de análisis. Por ahora, una palabra. |
| **vocabulario** | La lista de palabras distintas de todo el conjunto de documentos. |
| **vector** | La lista de números que representa a un documento. |
| **matriz documento-término** | La tabla con un documento por fila y una palabra por columna. |
| **bag of words** | Representar un documento por las palabras que contiene, sin orden. |
| **matriz esparsa** | Una matriz que guarda solo los valores distintos de cero. |
| **stopwords** | Palabras vacías que se descartan antes de contar. La lista la escribe alguien. |
| **TF** | Cuánto aparece una palabra en un documento. |
| **IDF** | Cuán rara es una palabra en el conjunto. |
| **n-grama** | Una secuencia de n palabras seguidas. |